In [1]:
# @title ⏱️ Premium Timer Settings (Auto-Reflect) { run: "auto" }
# Specify total time (The frame automatically expands even if it exceeds 1 hour)
Time_Limit_Hours = 0 # @param {type:"number"}
Time_Limit_Minutes = 8 # @param {type:"number"}
Time_Limit_Seconds = 00 # @param {type:"number"}

# Specify warning timing
Warning_Timing_Minutes = 1 # @param {type:"number"}
Warning_Timing_Seconds = 0 # @param {type:"number"}

from IPython.display import display, HTML
import uuid

# Unify all settings into seconds on the Python side
limit_sec = int(Time_Limit_Hours) * 3600 + int(Time_Limit_Minutes) * 60 + int(Time_Limit_Seconds)
warn_sec = int(Warning_Timing_Minutes) * 60 + int(Warning_Timing_Seconds)

# Generate a unique ID to prevent conflicts between different cells
unique_id = str(uuid.uuid4())[:8]

html_code = f"""
<!-- Outer container: Automatically expands based on content width (Min-width: 1150px) -->
<div id="timer-container-{unique_id}" style="text-align:center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin: 30px auto; width: max-content; min-width: 1150px; background-color: #f8f9fa; padding: 30px; border-radius: 45px; box-shadow: 0px 20px 5px rgba(0,0,0,0.02);">

    <!-- Control Area (Enlarged buttons with enough padding to prevent wrapping into 2 lines) -->
    <div style="margin-bottom: 35px; display: flex; justify-content: center; align-items: center; gap: 25px;">
        <button id="start-btn-{unique_id}" style="background-color: #10b981; color: white; border: none; padding: 15px 0; font-size: 20px; border-radius: 14px; cursor: pointer; font-weight: bold; width: 180px; transition: all 0.2s; box-shadow: 0 4px 6px rgba(16, 185, 129, 0.2); white-space: nowrap;">START</button>
        <button id="pause-btn-{unique_id}" style="background-color: #f59e0b; color: white; border: none; padding: 15px 0; font-size: 20px; border-radius: 14px; cursor: pointer; font-weight: bold; width: 180px; transition: all 0.2s; box-shadow: 0 4px 6px rgba(245, 158, 11, 0.2); white-space: nowrap;">PAUSE</button>
        <button id="reset-btn-{unique_id}" style="background-color: #ef4444; color: white; border: none; padding: 15px 0; font-size: 20px; border-radius: 14px; cursor: pointer; font-weight: bold; width: 180px; transition: all 0.2s; box-shadow: 0 4px 6px rgba(239, 68, 68, 0.2); white-space: nowrap;">RESET</button>
    </div>

    <!-- Extra Large Timer Display Panel (High aspect ratio and responsive width) -->
    <div id="display-panel-{unique_id}" style="font-family: 'SF Pro Display', 'Helvetica Neue', 'Courier New', monospace; font-weight: 800; padding: 90px 80px; border: 18px solid #3b82f6; border-radius: 45px; background-color: #ffffff; box-shadow: 0px 25px 50px rgba(0,0,0,0.15); transition: all 0.5s ease; display: inline-block; width: auto;">
        <!-- Keeps text size at 300px and uses monospace font variant to prevent layout shifting -->
        <div id="time-text-{unique_id}" style="font-size: 300px; color: #3b82f6; line-height: 0.9; letter-spacing: -2px; margin: 0; font-variant-numeric: tabular-nums; white-space: nowrap;">00:00</div>
        <div id="sub-text-{unique_id}" style="font-size: 46px; color: #6b7280; margin-top: 40px; font-weight: 600; letter-spacing: 4px;">READY</div>
    </div>
</div>

<script>
(function() {{
    const limitSec = {limit_sec};
    const warnSec = {warn_sec};

    let remaining = limitSec;
    let timerId = null;
    let bellWarnPlayed = false;
    let bellFinishPlayed = false;
    let lastExtraMin = 0;

    const startBtn = document.getElementById('start-btn-{unique_id}');
    const pauseBtn = document.getElementById('pause-btn-{unique_id}');
    const resetBtn = document.getElementById('reset-btn-{unique_id}');
    const displayPanel = document.getElementById('display-panel-{unique_id}');
    const timeText = document.getElementById('time-text-{unique_id}');
    const subText = document.getElementById('sub-text-{unique_id}');

    // Special Bell Sound: 2250Hz / 1.8s decay
    function playMetalBell(times) {{
        try {{
            const AudioContext = window.AudioContext || window.webkitAudioContext;
            if (!AudioContext) return;
            const ctx = new AudioContext();

            let i = 0;
            function trigger() {{
                if (i < times) {{
                    const now = ctx.currentTime;

                    const osc1 = ctx.createOscillator();
                    const gain1 = ctx.createGain();
                    osc1.type = 'triangle';
                    osc1.frequency.setValueAtTime(2250, now);
                    gain1.gain.setValueAtTime(0.25, now);
                    gain1.gain.exponentialRampToValueAtTime(0.001, now + 1.8);

                    const osc2 = ctx.createOscillator();
                    const gain2 = ctx.createGain();
                    osc2.type = 'sine';
                    osc2.frequency.setValueAtTime(4500, now);
                    gain2.gain.setValueAtTime(0.12, now);
                    gain2.gain.exponentialRampToValueAtTime(0.001, now + 1.0);

                    osc1.connect(gain1);
                    gain1.connect(ctx.destination);
                    osc2.connect(gain2);
                    gain2.connect(ctx.destination);

                    osc1.start(now);
                    osc1.stop(now + 1.8);
                    osc2.start(now);
                    osc2.stop(now + 1.8);

                    i++;
                    setTimeout(trigger, 800);
                }}
            }}
            trigger();
        }} catch (e) {{
            console.log("Audio Context Error:", e);
        }}
    }}

    function formatTime(sec) {{
        const absSec = Math.abs(sec);
        const s = absSec % 60;
        const m = Math.floor(absSec / 60) % 60;
        const h = Math.floor(absSec / 3600);

        const pad = (num) => String(num).padStart(2, '0');

        let timeStr = "";
        if (h > 0) {{
            timeStr = h + ":" + pad(m) + ":" + pad(s);
        }} else {{
            timeStr = pad(m) + ":" + pad(s);
        }}

        return sec < 0 ? "+" + timeStr : timeStr;
    }}

    function updateUI() {{
        timeText.textContent = formatTime(remaining);

        if (remaining >= 0) {{
            if (remaining <= warnSec && limitSec > 0) {{
                displayPanel.style.borderColor = "#f59e0b";
                timeText.style.color = "#f59e0b";
                subText.textContent = "ENDING SOON";
                subText.style.color = "#f59e0b";
            }} else {{
                displayPanel.style.borderColor = "#3b82f6";
                timeText.style.color = "#3b82f6";
                subText.textContent = "PRESENTATION";
                subText.style.color = "#6b7280";
            }}
        }} else {{
            displayPanel.style.borderColor = "#ef4444";
            timeText.style.color = "#ef4444";
            subText.textContent = "TIME OVER";
            subText.style.color = "#ef4444";
        }}
    }}

    function tick() {{
        remaining--;

        if (remaining === warnSec && !bellWarnPlayed && remaining > 0) {{
            playMetalBell(1);
            bellWarnPlayed = true;
        }}

        if (remaining === 0 && !bellFinishPlayed) {{
            playMetalBell(2);
            bellFinishPlayed = true;
        }}

        if (remaining < 0 && Math.abs(remaining) % 60 === 0) {{
            const currentExtra = Math.floor(Math.abs(remaining) / 60);
            if (currentExtra > lastExtraMin) {{
                playMetalBell(3);
                lastExtraMin = currentExtra;
            }}
        }}

        updateUI();
    }}

    startBtn.onclick = function() {{
        if (!timerId) {{
            if (remaining === limitSec) {{
                subText.textContent = "PRESENTATION";
            }}
            timerId = setInterval(tick, 1000);
        }}
    }};

    pauseBtn.onclick = function() {{
        if (timerId) {{
            clearInterval(timerId);
            timerId = null;
            subText.textContent = "PAUSED";
            subText.style.color = "#f59e0b";
        }}
    }};

    resetBtn.onclick = function() {{
        if (timerId) {{
            clearInterval(timerId);
            timerId = null;
        }}
        remaining = limitSec;
        bellWarnPlayed = false;
        bellFinishPlayed = false;
        lastExtraMin = 0;
        updateUI();
        subText.textContent = "READY";
        subText.style.color = "#6b7280";
    }};

    updateUI();
    subText.textContent = "READY";

}})();
</script>
"""

display(HTML(html_code))